[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_14_cross_entropy_full.ipynb)

# 🟡 Medium: Cross-Entropy: Smoothing & Padding Mask

*Training*
Problem 16 built the loss itself. This is the version training code actually
calls: **label smoothing** and a **padding mask**, over inputs of any rank.

$$\ell_i = -\sum_{c} q_{i,c} \log p_{i,c}, \qquad
p_{i,c} = \frac{e^{z_{i,c}}}{\sum_{k} e^{z_{i,k}}}$$

With smoothing $\alpha$ over $C$ classes the target distribution is
$q_{i,c} = (1-\alpha)\,\mathbb{1}[c = t_i] + \alpha/C$, so

$$\ell_i = (1-\alpha)\bigl(-\log p_{i,t_i}\bigr) \;+\; \frac{\alpha}{C}\sum_c \bigl(-\log p_{i,c}\bigr)$$

Return the **mean over the non-ignored positions only**.

### Rules
- Signature: `cross_entropy_loss(logits, targets, *, label_smoothing=0.0, ignore_index=-1)`
- `logits` is `(..., C)`, `targets` is `(...)` of integer class ids; the output is a **scalar**
- Banned: `jax.nn.log_softmax`, `jax.nn.softmax`, `jax.scipy.special.logsumexp`, `optax`
- Compute $\log p$ in one fused expression; never form $p$ and then take its log
- Positions where `targets == ignore_index` contribute **zero** loss and are excluded
  from the denominator; if every position is ignored, return `0.0` (not `nan`)
- No `if` on array values — the whole thing must work under `jit` and `vmap`

> Do **problem 16** first. The stability half is unchanged here, and this
> problem assumes you already have it.

### The interview angle
`ignore_index` is where candidates lose the plot. Pad-to-longest batching over
variable-length sequences routinely leaves a third or more of the positions as
`<pad>`. Average over all of them and two things go wrong: the loss is scaled
down by the pad fraction (so it silently changes meaning when the batch
composition changes), and the gradient actively teaches the model to predict
`<pad>`.

The second half is the index itself, and it is worth knowing exactly what JAX
does here because it does **not** raise. `jnp.take_along_axis` defaults to
`mode="fill"`, so a genuinely out-of-range sentinel like `-100` gathers `NaN` —
which then propagates through the mean and poisons the whole batch even though
you "masked" it afterwards. And `-1` does not go out of range at all: it wraps
round to the last class, so you get a plausible-looking wrong loss with nothing
to alert you. Both are fixed the same way — substitute a valid index *before*
the gather — but only if you knew there was something to fix.

The zero denominator is the other one. A batch where every position is padding
is not hypothetical — it happens on the last shard of an evaluation loop — and
`0/0` is `nan`, which then spreads to every parameter through the update. Clamp
the count with `jnp.maximum(n_valid, 1)`: the numerator is already `0`, so the
answer is `0.0`.

### Why smoothing is a blend, not a subtraction
$q$ has to stay a distribution. Spreading $\alpha$ over all $C$ classes —
*including* the true one, which keeps $1 - \alpha + \alpha/C$ — is what makes
$\sum_c q_{i,c} = 1$. Take the mass from the true class and hand it only to the
other $C-1$ and you get a different loss that still looks plausible; the test
suite checks the exact blend.

What it buys you: the gradient is now $p - q$ with $q$ bounded away from the
one-hot corner, so no logit is ever pushed to $+\infty$. The model stops
spending capacity on being *more* certain about tokens it already gets right,
which is why label smoothing shows up in the Transformer paper's recipe
($\alpha = 0.1$) alongside a *worse* perplexity but a better BLEU.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cross_entropy_loss(logits, targets, *, label_smoothing=0.0, ignore_index=-1):
    """Mean cross-entropy over the non-ignored positions.

    Args:
        logits:          (..., C) unnormalised scores
        targets:         (...) integer class ids
        label_smoothing: alpha in [0, 1); target is (1-a)*onehot + a/C
        ignore_index:    positions equal to this are masked out entirely

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# With no smoothing and no padding this is just problem 16 again.
print("uniform:", cross_entropy_loss(jnp.zeros((1, 3)), jnp.array([0])), "vs", jnp.log(3.0))

# Padding. Non-uniform logits, so the denominator genuinely matters.
logits = jax.random.normal(jax.random.key(0), (4, 5)) * 3.0
print("\nmean over the 2 real tokens:",
      float(cross_entropy_loss(logits, jnp.array([1, 2, -1, -1]), ignore_index=-1)))
print("mean over all 4 positions  :",
      float(cross_entropy_loss(logits, jnp.array([1, 2, 0, 0]))), " <- wrong denominator")

# Every position ignored -> 0.0, not nan.
print("all padding:", float(cross_entropy_loss(logits, jnp.full((4,), -1), ignore_index=-1)))

# Why the index must be sanitised BEFORE the gather: JAX never raises here.
print("\ngather at -1  ->", jnp.take_along_axis(logits, jnp.full((4, 1), -1), -1)[:, 0])
print("               ...that is column 4, read silently")
print("gather at -100->", jnp.take_along_axis(logits, jnp.full((4, 1), -100), -1)[:, 0])
print("               ...NaN, which masking afterwards will not remove")

# Smoothing penalises over-confidence.
sharp, t0 = jnp.array([[20.0, 0.0, 0.0]]), jnp.array([0])
print("\nsharp, alpha=0.0:", float(cross_entropy_loss(sharp, t0)))
print("sharp, alpha=0.1:", float(cross_entropy_loss(sharp, t0, label_smoothing=0.1)))

# (B, T, C) sequences, the shape this actually gets called on.
lg = jax.random.normal(jax.random.key(1), (2, 6, 7))
tg = jax.random.randint(jax.random.key(2), (2, 6), 0, 7).at[:, -2:].set(-1)
print("\n(B, T, C) with 4 padded positions:",
      float(cross_entropy_loss(lg, tg, label_smoothing=0.1, ignore_index=-1)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_entropy_full")

# hint("cross_entropy_full")      # stuck? nudge without the answer
# solution("cross_entropy_full")  # spoiler: the reference implementation
# status()                        # your dashboard across all problems